# Unit tests
Test one unit with fast, deterministic inputs.


In [ ]:
def calculate_total(price: float, quantity: int) -> float:
    return price * quantity

assert calculate_total(2.5, 4) == 10.0
print("passed")


## Polished version
Test business behavior through an interface-backed fake.


In [ ]:
from dataclasses import dataclass
from typing import Protocol

@dataclass(frozen=True)
class Task:
    id: int
    completed: bool = False

class TaskRepository(Protocol):
    def get(self, task_id: int) -> Task | None: ...
    def save(self, task: Task) -> None: ...

class FakeTaskRepository:
    def __init__(self, tasks: list[Task]) -> None:
        self.tasks = {task.id: task for task in tasks}
    def get(self, task_id: int) -> Task | None:
        return self.tasks.get(task_id)
    def save(self, task: Task) -> None:
        self.tasks[task.id] = task

class TaskService:
    def __init__(self, repository: TaskRepository) -> None:
        self.repository = repository
    def complete(self, task_id: int) -> Task:
        task = self.repository.get(task_id)
        if task is None:
            raise LookupError("task not found")
        completed = Task(task.id, completed=True)
        self.repository.save(completed)
        return completed

repository = FakeTaskRepository([Task(1)])
result = TaskService(repository).complete(1)
assert result.completed is True
assert repository.get(1) == result
print("passed")
